# VibeIQ Audience Intelligence Engine
End-to-end synthetic streaming audience analytics: churn, LTV, next action and segmentation.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, roc_auc_score, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


In [ ]:
ROOT=Path('..')
df=pd.read_csv(ROOT/'data/users.csv')
events=pd.read_csv(ROOT/'data/streaming_events.csv')
df.head()


## EDA
Inspect churn rate, engagement distributions, segments and regional/language patterns.


In [ ]:
print(df.shape)
print(df[['sessions_7d','listening_minutes_7d','skip_rate','save_rate','discovery_rate','churn','ltv']].describe())
print(df.groupby('language').churn.mean().sort_values(ascending=False))


## Churn model
The target is observed churn. This is supervised learning; behavioral inputs without an outcome label cannot teach the classifier.


In [ ]:
FEATURES=['sessions_7d','session_change_7d','listening_minutes_7d','listening_change_7d','skip_rate','save_rate','share_rate','playlist_rate','search_rate','discovery_rate','unique_artists_7d','unique_genres_7d','days_active_14d','avg_session_minutes','subscription_age_days','support_tickets_30d','night_listening_share','completion_rate']
X=df[FEATURES]; y=df.churn
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
model=RandomForestClassifier(n_estimators=120,max_depth=12,min_samples_leaf=4,class_weight='balanced',random_state=42,n_jobs=-1)
model.fit(Xtr,ytr)
p=model.predict_proba(Xte)[:,1]
print(classification_report(yte,p>=.5))
print('ROC-AUC:',roc_auc_score(yte,p))


## LTV regression
LTV is modeled on a log-transformed target because user value is right-skewed.


In [ ]:
reg=RandomForestRegressor(n_estimators=120,max_depth=14,min_samples_leaf=4,random_state=42,n_jobs=-1)
reg.fit(Xtr,np.log1p(df.loc[Xtr.index,'ltv']))
pred_ltv=np.expm1(reg.predict(Xte))
print('MAE:',mean_absolute_error(df.loc[Xte.index,'ltv'],pred_ltv))
print('R2:',r2_score(df.loc[Xte.index,'ltv'],pred_ltv))


## Audience segmentation
Standardize behavior and use K-Means to discover behavioral groups.


In [ ]:
scaler=StandardScaler()
Z=scaler.fit_transform(X)
km=KMeans(n_clusters=5,n_init=10,random_state=42)
df['cluster']=km.fit_predict(Z)
print(df.groupby('cluster')[FEATURES[:8]].mean().round(2))


## Business output
Combine churn probability and LTV to prioritize retention: high churn + high value is the most urgent population.


In [ ]:
df['churn_probability']=model.predict_proba(X)[:,1]
high_value=df.ltv.quantile(.75)
priority=df[(df.churn_probability>=.72)&(df.ltv>=high_value)].sort_values('churn_probability',ascending=False)
priority[['user_id','churn_probability','ltv']].head(20)
